In [1]:
# Cell 1: Imports
import time
import numpy as np
import matplotlib.pyplot as plt

# TensorFlow
import tensorflow as tf
from tensorflow.keras import layers, models, datasets

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Check GPU availability
print("TensorFlow GPU:", tf.config.list_physical_devices('GPU'))
print("PyTorch GPU Available:", torch.cuda.is_available())

TensorFlow GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
PyTorch GPU Available: True


In [2]:
# Cell 2: TensorFlow MLP
# Load and normalize data
(X_train_tf, y_train_tf), (X_test_tf, y_test_tf) = datasets.cifar10.load_data()
X_train_tf, X_test_tf = X_train_tf / 255.0, X_test_tf / 255.0

# Build MLP Model
tf_mlp = models.Sequential([
    layers.Flatten(input_shape=(32, 32, 3)),
    layers.Dense(512, activation='relu'),
    layers.Dense(256, activation='relu'),
    layers.Dense(10)
])

tf_mlp.compile(optimizer='adam',
               loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
               metrics=['accuracy'])

# Train and time the model
start_time = time.time()
history_tf_mlp = tf_mlp.fit(X_train_tf, y_train_tf, epochs=10, batch_size=64, validation_split=0.1, verbose=1)
tf_mlp_time = time.time() - start_time

# Evaluate
tf_mlp_loss, tf_mlp_acc = tf_mlp.evaluate(X_test_tf, y_test_tf, verbose=0)
print(f"\nTensorFlow MLP - Training Time: {tf_mlp_time:.2f} seconds | Test Accuracy: {tf_mlp_acc:.4f}")

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 23s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.3236 - loss: 1.8843 - val_accuracy: 0.3594 - val_loss: 1.8017
Epoch 2/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.3982 - loss: 1.6835 - val_accuracy: 0.4080 - val_loss: 1.6331
Epoch 3/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.4325 - loss: 1.5879 - val_accuracy: 0.4308 - val_loss: 1.5952
Epoch 4/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.4514 - loss: 1.5357 - val_accuracy: 0.4412 - val_loss: 1.5793
Epoch 5/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.4649 - loss: 1.4945 - val_accuracy: 0.4496 - val_loss: 1.5393
Epoch 6/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.4772 - loss: 1.4617 - val_accuracy: 0.4644 - val_loss: 1.5333
Epoch 7/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.4882 - loss: 1.4322 - val_accuracy: 0.4374 - val_loss: 1.6118
Epoch 8/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.4992 - loss: 1.4061 - val_accuracy: 0.

In [3]:
# Cell 3: PyTorch MLP
# Data transforms and loading
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = DataLoader(testset, batch_size=64, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Build MLP Model
class PyTorchMLP(nn.Module):
    def __init__(self):
        super(PyTorchMLP, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 32 * 3, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)

torch_mlp = PyTorchMLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(torch_mlp.parameters())

# Train and time the model
start_time = time.time()
torch_mlp.train()
for epoch in range(10):
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = torch_mlp(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
torch_mlp_time = time.time() - start_time

# Evaluate
torch_mlp.eval()
correct, total = 0, 0
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = torch_mlp(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

torch_mlp_acc = correct / total
print(f"PyTorch MLP - Training Time: {torch_mlp_time:.2f} seconds | Test Accuracy: {torch_mlp_acc:.4f}")

100%|██████████| 170M/170M [00:25<00:00, 6.58MB/s]


PyTorch MLP - Training Time: 129.93 seconds | Test Accuracy: 0.5214


In [6]:
# Cell 4: TensorFlow CNN
tf_cnn = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10)
])

tf_cnn.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

start_time = time.time()
history_tf_cnn = tf_cnn.fit(X_train_tf, y_train_tf, epochs=10, batch_size=64, validation_split=0.1, verbose=1)
tf_cnn_time = time.time() - start_time

tf_cnn_loss, tf_cnn_acc = tf_cnn.evaluate(X_test_tf, y_test_tf, verbose=0)
print(f"\nTensorFlow CNN - Training Time: {tf_cnn_time:.2f} seconds | Test Accuracy: {tf_cnn_acc:.4f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.4414 - loss: 1.5515 - val_accuracy: 0.5188 - val_loss: 1.3346
Epoch 2/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.5827 - loss: 1.1930 - val_accuracy: 0.6194 - val_loss: 1.1012
Epoch 3/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6339 - loss: 1.0477 - val_accuracy: 0.6304 - val_loss: 1.0769
Epoch 4/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6630 - loss: 0.9663 - val_accuracy: 0.6602 - val_loss: 0.9870
Epoch 5/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6870 - loss: 0.9001 - val_accuracy: 0.6718 - val_loss: 0.9524
Epoch 6/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7056 - loss: 0.8466 - val_accuracy: 0.7046 - val_loss: 0.8803
Epoch 7/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.7246 - loss: 0.7989 - val_accuracy: 0.6908 - val_loss: 0.8986
Epoch 8/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7413 - loss: 0.7532 - val_accuracy: 0.

In [4]:
# Cell 5: PyTorch CNN
class PyTorchCNN(nn.Module):
    def __init__(self):
        super(PyTorchCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        self.fc1 = nn.Linear(64 * 6 * 6, 64) # Standard output resolution tracking
        self.fc2 = nn.Linear(64, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 64 * 6 * 6)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

torch_cnn = PyTorchCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(torch_cnn.parameters())

start_time = time.time()
torch_cnn.train()
for epoch in range(10):
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = torch_cnn(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
torch_cnn_time = time.time() - start_time

torch_cnn.eval()
correct, total = 0, 0
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = torch_cnn(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

torch_cnn_acc = correct / total
print(f"PyTorch CNN - Training Time: {torch_cnn_time:.2f} seconds | Test Accuracy: {torch_cnn_acc:.4f}")

PyTorch CNN - Training Time: 138.30 seconds | Test Accuracy: 0.7006


In [7]:
# Cell 6: Summary Table
import pandas as pd

data = {
    "Model": ["Simple MLP", "Simple MLP", "CNN", "CNN"],
    "Framework": ["TensorFlow", "PyTorch", "TensorFlow", "PyTorch"],
    "Training Time (s)": [tf_mlp_time, torch_mlp_time, tf_cnn_time, torch_cnn_time],
    "Accuracy (%)": [tf_mlp_acc * 100, torch_mlp_acc * 100, tf_cnn_acc * 100, torch_cnn_acc * 100]
}

df = pd.DataFrame(data)
print(df.to_string(index=False))

     Model  Framework  Training Time (s)  Accuracy (%)
Simple MLP TensorFlow          33.951674     47.780001
Simple MLP    PyTorch         129.934392     52.140000
       CNN TensorFlow          39.814700     67.690003
       CNN    PyTorch         138.303166     70.060000


### Step 1: Configuration
Set your GitHub credentials. It is recommended to store your token in Colab's secret manager (the key icon on the left) as `GITHUB_TOKEN`.

In [13]:
import os
from google.colab import userdata

# 1. Configuration
USER_NAME = "Oak-ke"
USER_EMAIL = "mwamipayments@gmail.com"
REPO_NAME = "cifar10-tf-pytorch-comparison"

# 2. Securely get the token from Colab Secrets
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    print("Error: Please add 'GITHUB_TOKEN' to the Secrets (key icon) on the left!")

# 3. Configure Git
!git config --global user.email "{USER_EMAIL}"
!git config --global user.name "{USER_NAME}"

print("Git configured successfully using Secrets.")

Git configured for: Oak-ke


### Step 2: Create the README.md

In [17]:
readme_content = """
# CIFAR-10: TensorFlow vs. PyTorch Comparison

This project benchmarks TensorFlow and PyTorch using CIFAR-10. It compares MLP and CNN architectures on training speed and accuracy. Results show TensorFlow trained significantly faster (~40s vs ~138s), while PyTorch achieved slightly higher CNN accuracy (~70%). It is an assignment for my Deep Learning Class.

This notebook provides a side-by-side comparison of TensorFlow and PyTorch for classifying the CIFAR-10 dataset.

## Models Included
1. **Simple MLP**: A basic Multi-Layer Perceptron.
2. **CNN**: A Convolutional Neural Network with max-pooling.

## Key Metrics
- Training time (seconds)
- Test accuracy (%)

## Performance Summary
A summary table is generated at the end of the notebook to compare the training efficiency and final accuracy across both frameworks."""


with open("README.md", "w") as f:
    f.write(readme_content.strip())

### Step 3: Initialize and Push
This will create a local repo, add the file, and push to GitHub.

In [18]:
# Initialize and Push
!git init
!git config user.email "{USER_EMAIL}"
!git config user.name "{USER_NAME}"
!git add README.md
!git commit -m "feat: initial commit with readme"
!git branch -M main

# Remote URL with Token authentication
remote_url = f"https://{GITHUB_TOKEN}@github.com/{USER_NAME}/{REPO_NAME}.git"

# Try to remove origin if it exists, then add and push
!git remote remove origin 2>/dev/null
!git remote add origin {remote_url}
!git push -u origin main

Reinitialized existing Git repository in /content/.git/
[main 079447c] feat: initial commit with readme
 1 file changed, 14 insertions(+), 1 deletion(-)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (3/3), 733 bytes | 733.00 KiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/Oak-ke/cifar10-tf-pytorch-comparison.git
   be70c93..079447c  main -> main
Branch 'main' set up to track remote branch 'main' from 'origin'.


In [ ]:
import json
import requests
from google.colab import _message

# 1. Clear the failed/blocked commit to start fresh
!git reset --soft HEAD~1

# 2. Refresh the notebook content (now with the token removed from code)
notebook_json = _message.blocking_request('get_ipynb', request='', timeout_sec=30)
filename = "cifar10_comparison.ipynb"
with open(filename, 'w') as f:
    json.dump(notebook_json['ipynb'], f)

# 3. Re-add and push
!git add {filename}
!git commit -m "feat: add notebook file securely"

# Use the token from the variable we just loaded via userdata
remote_url = f"https://{GITHUB_TOKEN}@github.com/{USER_NAME}/{REPO_NAME}.git"
!git remote set-url origin {remote_url}
!git push origin main --force